### Ejercicio 10: Re-ranking

Objetivo: Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.


#### Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia)


In [1]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

c:\Users\wwwed\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\beir\util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets\scifact.zip: 100%|██████████| 2.69M/2.69M [00:13<00:00, 216kiB/s] 


'../data/beir_datasets\\scifact'

In [3]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

100%|██████████| 5183/5183 [00:00<00:00, 115584.82it/s]


In [4]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [5]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [6]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [7]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


#### Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [9]:
from rank_bm25 import BM25Okapi
from beir.retrieval.evaluation import EvaluateRetrieval

# 1. Preparar el corpus (tokenización básica)
corpus_ids = list(corpus.keys())
tokenized_corpus = [corpus[doc_id]['text'].lower().split() for doc_id in corpus_ids]

# 2. Inicializar BM25
bm25 = BM25Okapi(tokenized_corpus)

# 3. Realizar retrieval para todas las queries (Top 100)
top_k = 100
bm25_results = {}

# CORRECCIÓN: 'query_text' ya es directamente el string
for qid, query_text in queries.items():
    tokenized_query = query_text.lower().split()
    
    # Obtener scores y ordenar
    scores = bm25.get_scores(tokenized_query)
    top_k_indices = scores.argsort()[-top_k:][::-1]
    
    # Guardar resultados en el formato requerido por BEIR: {qid: {doc_id: score}}
    bm25_results[qid] = {corpus_ids[i]: float(scores[i]) for i in top_k_indices}

# 4. Evaluación inicial
evaluator = EvaluateRetrieval()
ndcg, _map, recall, precision = evaluator.evaluate(qrels, bm25_results, k_values=[10])

print("\n--- Resultados Baseline (BM25) ---")
print(f"nDCG@10:  {ndcg['NDCG@10']:.4f}")
print(f"Recall@10: {recall['Recall@10']:.4f}")


--- Resultados Baseline (BM25) ---
nDCG@10:  0.5438
Recall@10: 0.6688


#### Parte 3. Implementación del re-ranking cross-encoder

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [11]:
from sentence_transformers import CrossEncoder

# Cargar un modelo ligero pero potente para re-ranking
ce_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

ce_results = {}

print("Iniciando Re-ranking con Cross-Encoder...")
for qid, docs in bm25_results.items():
    query_text = queries[qid]
    ce_results[qid] = {}
    
    # Preparar los pares [Query, Documento]
    doc_ids = list(docs.keys())
    cross_inp = [[query_text, corpus[doc_id]['title'] + " " + corpus[doc_id]['text']] for doc_id in doc_ids]
    
    # Predecir nuevos scores
    ce_scores = ce_model.predict(cross_inp)
    
    for i, doc_id in enumerate(doc_ids):
        ce_results[qid][doc_id] = float(ce_scores[i])

# Identificar cambios en el Top 10 para la query seleccionada ("133")
qid_test = "133"
top_10_bm25 = [doc for doc, score in sorted(bm25_results[qid_test].items(), key=lambda x: x[1], reverse=True)[:10]]
top_10_ce = [doc for doc, score in sorted(ce_results[qid_test].items(), key=lambda x: x[1], reverse=True)[:10]]

print(f"\n--- Cambios de posición para Query {qid_test} ---")
print(f"Top 10 BM25: {top_10_bm25}")
print(f"Top 10 Cross-Encoder: {top_10_ce}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4231.30it/s]


Iniciando Re-ranking con Cross-Encoder...

--- Cambios de posición para Query 133 ---
Top 10 BM25: ['26688294', '37964706', '9507605', '5270265', '12785130', '45764440', '86694016', '12640810', '5821617', '17934082']
Top 10 Cross-Encoder: ['35660758', '12640810', '16280642', '36345185', '6969753', '9507605', '86694016', '19752008', '17934082', '21295300']


#### Parte 4. Implementación del re-ranking LTR

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10




In [14]:
import lightgbm as lgb
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# 1. Definir función de extracción de features
def extract_features(qid, doc_id, bm25_score):
    query_text = queries[qid]
    doc_text = corpus[doc_id]['text']
    # Features básicas: score BM25, longitud query, longitud doc
    return [bm25_score, len(query_text.split()), len(doc_text.split())]

X, y, group = [], [], []
ltr_results = {}

# 2. Preparar el dataset para el ranker (usando los candidatos de BM25)
for qid, docs in bm25_results.items():
    group_count = 0
    for doc_id, bm25_score in docs.items():
        features = extract_features(qid, doc_id, bm25_score)
        
        # Obtener relevancia real (1, 2, etc.) o 0 si no es relevante
        rel = qrels.get(qid, {}).get(doc_id, 0)
        
        X.append(features)
        y.append(rel)
        group_count += 1
    
    group.append(group_count)

# 3. Entrenar el modelo LambdaRank con LightGBM
ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    importance_type='gain',
    verbose=-1
)
ranker.fit(X, y, group=group)

# 4. Re-rankear usando el modelo LTR
idx = 0
for qid, docs in bm25_results.items():
    ltr_results[qid] = {}
    doc_ids = list(docs.keys())
    
    # Extraer características para predicción
    X_pred = [extract_features(qid, doc_id, docs[doc_id]) for doc_id in doc_ids]
    
    # Predecir
    ltr_scores = ranker.predict(X_pred)
    
    for i, doc_id in enumerate(doc_ids):
        ltr_results[qid][doc_id] = float(ltr_scores[i])

# Ver cambios para el LTR
top_10_ltr = [doc for doc, score in sorted(ltr_results[qid_test].items(), key=lambda x: x[1], reverse=True)[:10]]
print(f"\nTop 10 LTR: {top_10_ltr}")


Top 10 LTR: ['26688294', '12640810', '17934082', '15893330', '29073751', '5270265', '9063688', '4399311', '37964706', '19752008']


#### Parte 5. Evaluación post re-ranking

Calcular métricas:

* nDCG@10
* MAP
* Recall@10


In [13]:
print("\n===========================================")
print("=== EVALUACIÓN FINAL DE LOS MODELOS ===")
print("===========================================")

# 1. Evaluar Cross-Encoder
ndcg_ce, map_ce, recall_ce, precision_ce = evaluator.evaluate(qrels, ce_results, k_values=[10])

# 2. Evaluar LTR
ndcg_ltr, map_ltr, recall_ltr, precision_ltr = evaluator.evaluate(qrels, ltr_results, k_values=[10])

# 3. Consolidar resultados para visualización clara
import pandas as pd

resultados_evaluacion = {
    "Modelo": ["BM25 (Baseline)", "Cross-Encoder", "LTR (LightGBM)"],
    "nDCG@10": [ndcg['NDCG@10'], ndcg_ce['NDCG@10'], ndcg_ltr['NDCG@10']],
    "MAP@10":  [_map['MAP@10'], map_ce['MAP@10'], map_ltr['MAP@10']],
    "Recall@10": [recall['Recall@10'], recall_ce['Recall@10'], recall_ltr['Recall@10']]
}

df_resultados = pd.DataFrame(resultados_evaluacion)
print(df_resultados.to_string(index=False))


=== EVALUACIÓN FINAL DE LOS MODELOS ===
         Modelo  nDCG@10  MAP@10  Recall@10
BM25 (Baseline)  0.54384 0.49926    0.66883
  Cross-Encoder  0.64144 0.60471    0.73794
 LTR (LightGBM)  0.77236 0.76567    0.77994
